# 14 — MTCNN-croppa mustasch-detektorns träningsdata
**Fynd:** `data/dataset_v3` (mustache/clean) är fortfarande i rått CelebA-format (178×218), aldrig MTCNN-croppat — exakt samma train/inference-mismatch som löstes för epic-modellen i `08`, men aldrig fixat för den binära mustasch-detektorn. Appen matar den alltid med MTCNN-croppade bilder (`image_size=178, margin=40`) vid inferens, så det är format den faktiskt ska tränas på.

**OBS: destruktiv operation.** Backup tas först automatiskt — verifiera att den lyckas innan du fortsätter.

In [2]:
import os
import shutil
import numpy as np
from PIL import Image
from facenet_pytorch import MTCNN

DATA_DIR   = 'data/dataset_v3'
BACKUP_DIR = 'data/dataset_v3_backup_precrop'

mtcnn = MTCNN(image_size=178, margin=40, post_process=False)

print('MTCNN redo. image_size=178, margin=40 — matchar app.py exakt.')

[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


MTCNN redo. image_size=178, margin=40 — matchar app.py exakt.


## Steg 1 — Backup

In [ ]:
if os.path.exists(BACKUP_DIR):
    print(f'Backup finns redan på {BACKUP_DIR} — hoppar över kopiering.')
else:
    print(f'Kopierar {DATA_DIR} -> {BACKUP_DIR} ...')
    shutil.copytree(DATA_DIR, BACKUP_DIR)
    print('Backup klar!')

def count_files_recursive(folder):
    return sum(1 for _, _, fnames in os.walk(folder) for f in fnames
               if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')))

print(f'Original: {count_files_recursive(DATA_DIR)} bilder')
print(f'Backup:   {count_files_recursive(BACKUP_DIR)} bilder')

## Steg 2 — Samla alla filer

In [ ]:
def collect_files(folder):
    paths = []
    for root, dirs, files in os.walk(folder):
        for f in files:
            if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
                paths.append(os.path.join(root, f))
    return paths

all_files = collect_files(DATA_DIR)
print(f'Totalt {len(all_files)} bilder att processa (tar några minuter för ~12 000 bilder).')

## Steg 3 — MTCNN-croppa och skriv över
Om inget ansikte hittas (kan hända på en del CelebA-bilder) flyttas filen till en granskningsmapp istället för att tystas bort.

In [ ]:
NO_FACE_DIR = 'data/dataset_v3_no_face_found'
os.makedirs(NO_FACE_DIR, exist_ok=True)

success_count = 0
no_face_count = 0
error_count = 0

for i, path in enumerate(all_files):
    try:
        img = Image.open(path).convert('RGB')
        face = mtcnn(img)

        if face is not None:
            arr = face.permute(1, 2, 0).numpy().astype(np.uint8)
            cropped = Image.fromarray(arr)
            cropped.save(path)
            success_count += 1
        else:
            dest = os.path.join(NO_FACE_DIR, os.path.basename(path))
            shutil.move(path, dest)
            no_face_count += 1

    except Exception as e:
        print(f'Fel på {path}: {e}')
        error_count += 1

    if (i + 1) % 500 == 0:
        print(f'{i + 1}/{len(all_files)} klara...')

print('\nKlart!')
print(f'Croppade och sparade: {success_count}')
print(f'Inget ansikte hittat (flyttade till {NO_FACE_DIR}): {no_face_count}')
print(f'Fel: {error_count}')

## Steg 4 — Stickprovskontroll

In [ ]:
import random
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 6, figsize=(18, 3))

for ax in axes:
    cls = random.choice(['mustache', 'clean'])
    files = collect_files(os.path.join(DATA_DIR, cls))
    f = random.choice(files)
    img = Image.open(f)
    ax.imshow(img)
    ax.set_title(f'{cls}\n{img.size}')
    ax.axis('off')

plt.tight_layout()
plt.show()

## Nästa steg
Kör om `02_train_mustache_model.ipynb` på den nu MTCNN-konsekventa datan, och testa sedan i `app.py` igen — testa specifikt `bäckis.jpg` (gråmustasch) och kvinnobilden som gav falsklarm, plus några vanliga glasögonbilder.

## Steg 5 — Granska felklassificerade bilder
Innan vi testar live i appen: är de kvarvarande felen genuina modellmissar, eller dolda felmärkningar i CelebA? Samma typ av etikettrensning vi gjorde tidigare i projektet för den här modellen.

In [4]:
import tensorflow as tf

mustache_model = tf.keras.models.load_model('models/mustache_detector_3.keras')

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset='validation',
    seed=42,
    image_size=(178, 178),
    batch_size=32,
    label_mode='binary',
    shuffle=False
)

class_names = val_ds.class_names  # alfabetisk: ['clean', 'mustache']
file_paths = val_ds.file_paths

y_true, y_pred, probs = [], [], []
for images, labels in val_ds:
    preds = mustache_model.predict(images, verbose=0).flatten()
    y_true.extend(labels.numpy().flatten().astype(int))
    y_pred.extend((preds >= 0.5).astype(int))
    probs.extend(preds)

import numpy as np
y_true = np.array(y_true)
y_pred = np.array(y_pred)
probs = np.array(probs)

assert len(file_paths) == len(y_true)
print(f'{len(file_paths)} valideringsbilder, {class_names}')

Found 11924 files belonging to 2 classes.
Using 2384 files for validation.
2384 valideringsbilder, ['clean', 'mustache']


2026-06-26 18:22:45.922605: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [3]:
import matplotlib.pyplot as plt
from PIL import Image

# Byt mellan 'clean' (sant clean, gissat mustache) och 'mustache' (sant mustache, gissat clean)
TRUE_LABEL = 'mustache'  # eller 'clean'
true_idx = class_names.index(TRUE_LABEL)
wrong_idx = 1 - true_idx

matching = [
    i for i in range(len(y_true))
    if y_true[i] == true_idx and y_pred[i] == wrong_idx
]

print(f'{len(matching)} felklassificerade bilder med sann etikett "{TRUE_LABEL}".')

n = min(len(matching), 40)
cols = 5
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(3.5 * cols, 3.5 * rows))
axes = np.array(axes).flatten()

for ax, i in zip(axes, matching[:n]):
    img = Image.open(file_paths[i])
    ax.imshow(img)
    ax.set_title(f'{os.path.basename(file_paths[i])}\nprob={probs[i]:.3f}', fontsize=8)
    ax.axis('off')

for ax in axes[n:]:
    ax.axis('off')

plt.tight_layout()
plt.show()

NameError: name 'class_names' is not defined